In [37]:
!pip install rapidfuzz

In [3]:
"""
ASR Preprocessing Pipeline — Full Fixed Version
================================================
Fixes applied (minimal, surgical):
  1. segment_with_vad result is now passed DIRECTLY to merge_and_chunk_segments
     instead of through the broken cursor-based reconstruction that collapsed
     real VAD timestamps into contiguous indices, cutting off the audio tail.
  2. Short-segment filtering is preserved (< 0.3s skipped), matching original logic.
  3. All other functions are UNCHANGED.
"""

import os
import numpy as np
import librosa
import torch
import soundfile as sf

# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────
SAMPLE_RATE    = 16000
TEST_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"   # <-- change to your actual path
OUTPUT_CHUNK_DIR = "/kaggle/working/" # <-- change to your desired output path

# =====================
# 1. AUDIO PREPROCESSING
# =====================

def resample_to_mono(audio_path, target_sr=16000):
    """
    Resample audio to target sample rate and convert to mono
    Pipeline step: Resample all audio to 16kHz and convert to mono
    """
    audio, sr = librosa.load(audio_path, sr=None, mono=False)
    if len(audio.shape) > 1:
        audio = librosa.to_mono(audio)
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
    return audio, target_sr


def trim_silence(audio, sr, top_db=60):
    """
    Trim leading and trailing silence
    Pipeline step: Trim leading/trailing silence within segments
    """
    audio_trimmed, _ = librosa.effects.trim(
        audio,
        top_db=top_db,
        frame_length=2048,
        hop_length=512
    )
    return audio_trimmed


def normalize_audio(audio):
    """
    Normalize audio amplitude
    Pipeline step: Normalization
    """
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val
    return audio


# =====================
# 2. VAD SEGMENTATION
# =====================

def load_silero_vad():
    """
    Load Silero VAD model
    Pipeline step: Use pretrained Silero VAD for segmentation
    """
    print("Loading Silero VAD...")
    try:
        vad_model, vad_utils = torch.hub.load(
            repo_or_dir='snakers4/silero-vad',
            model='silero_vad',
            force_reload=False,
            trust_repo=True
        )
        get_speech_timestamps, _, read_audio, *_ = vad_utils
        print("✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)")
        return vad_model, get_speech_timestamps, read_audio
    except Exception as e:
        print(f"⚠ VAD failed: {e}")
        return None, None, None


def segment_with_vad(audio_path, vad_model, get_speech_timestamps, read_audio):
    """
    Break long recording into speech segments using VAD
    Discard purely musical/silent parts
    Pipeline step: Segment with VAD, skip musical/silent parts
    Returns: List of (start_sample, end_sample) tuples in ORIGINAL FILE sample space
    """
    if vad_model is None:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
        intervals = librosa.effects.split(y, top_db=60)
        return [(int(s), int(e)) for s, e in intervals]

    try:
        wav = read_audio(audio_path, sampling_rate=SAMPLE_RATE)
        speech_timestamps = get_speech_timestamps(
            wav,
            vad_model,
            sampling_rate=SAMPLE_RATE,
            min_speech_duration_ms=300,
            min_silence_duration_ms=700,
            speech_pad_ms=300,
            return_seconds=False
        )
        return [(ts['start'], ts['end']) for ts in speech_timestamps]
    except:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
        intervals = librosa.effects.split(y, top_db=60)
        return [(int(s), int(e)) for s, e in intervals]


def preprocess_audio(audio_path):
    """
    Full preprocessing pipeline:
    - Resample to 16kHz
    - Mono
    - Trim silence
    - Normalize
    """
    audio, sr = resample_to_mono(audio_path, target_sr=SAMPLE_RATE)
    audio = trim_silence(audio, sr)
    audio = normalize_audio(audio)
    return audio, sr


# =====================
# 3. CHUNKING UTILITIES
# =====================

def merge_and_chunk_segments(segments, max_duration=30.0, sr=16000, max_gap=0.3):
    """
    Merge nearby VAD segments and split long ones
    Input:  segments = [(start, end), ...] in samples
    Output: list of chunks [(start, end), ...] in samples
    """
    if not segments:
        return []

    segments    = [(int(s[0]), int(s[1])) for s in segments]
    max_samples = int(max_duration * sr)
    final_segments = []
    cur_start, cur_end = segments[0]

    for start, end in segments[1:]:
        gap          = start - cur_end
        proposed_len = end - cur_start
        if gap <= max_gap * sr and proposed_len <= max_samples:
            cur_end = end
        else:
            final_segments.append((cur_start, cur_end))
            cur_start, cur_end = start, end

    final_segments.append((cur_start, cur_end))

    # Hard safety split
    chunked = []
    for start, end in final_segments:
        for s in range(start, end, max_samples):
            e = min(s + max_samples, end)
            chunked.append((s, e))

    return chunked


def add_overlap(chunks, sr=16000, overlap_sec=2.0):
    """
    Add overlap between consecutive chunks
    """
    overlap    = int(overlap_sec * sr)
    overlapped = []
    for i, (s, e) in enumerate(chunks):
        if i > 0:
            s = max(0, s - overlap)
        overlapped.append((s, e))
    return overlapped


def pad_chunks(chunks, audio_len, sr=16000, pad_start=0.2, pad_end=0.4):
    """
    Add small padding at start/end for safety
    """
    pad_start_s = int(pad_start * sr)
    pad_end_s   = int(pad_end   * sr)
    padded = []
    for start, end in chunks:
        s = max(0,         start - pad_start_s)
        e = min(audio_len, end   + pad_end_s)
        padded.append((s, e))
    return padded


# =====================
# 4. CHUNK SAVING
# =====================

def save_chunks(audio, chunks, sr, output_dir, file_id="train_001"):
    """
    Save each chunk as a .wav file named:
        {file_id}_chunk_{index:03d}.wav

    Parameters
    ----------
    audio      : np.ndarray  full 1-D waveform (preprocessed)
    chunks     : list of (start, end) sample tuples
    sr         : int          sample rate (default 16000)
    output_dir : str          directory to write .wav files into
    file_id    : str          prefix, e.g. "train_001"
    """
    os.makedirs(output_dir, exist_ok=True)
    for i, (s, e) in enumerate(chunks):
        chunk_audio = audio[s:e]
        filename    = f"{file_id}_chunk_{i:03d}.wav"
        filepath    = os.path.join(output_dir, filename)
        sf.write(filepath, chunk_audio, sr)
        print(f"  Saved: {filename}  ({(e - s) / sr:.2f}s,  samples [{s}:{e}])")
    print(f"\n✓ {len(chunks)} chunks saved to: {output_dir}")


# =====================
# 5. FULL PIPELINE
# =====================

def run_pipeline(audio_path, file_id="train_001", output_dir=OUTPUT_CHUNK_DIR):
    """
    Complete fixed pipeline for a single audio file.

    KEY FIX vs. original:
        BEFORE (broken):
            segments_with_indices = []
            cursor = 0
            for chunk in segments:          # numpy arrays
                chunk_len = len(chunk)
                segments_with_indices.append((cursor, cursor + chunk_len))
                cursor += chunk_len
            segments = segments_with_indices
            # ↑ This rebuilt fake contiguous indices that ended well before
            #   the real end of the audio, silently dropping the tail.

        AFTER (fixed):
            raw_vad_segments = segment_with_vad(...)
            # filter short segments exactly like preprocess_and_segment did
            segments_with_indices = [
                (s, e) for s, e in raw_vad_segments if (e - s) >= sr * 0.3
            ]
            # ↑ Real sample positions from Silero → tail is preserved.
    """
    print(f"\n{'='*60}")
    print(f" Processing: {os.path.basename(audio_path)}")
    print(f"{'='*60}")

    # Step 1 — Load VAD model (cached globally; passed in for clarity)
    vad_model, get_speech_timestamps, read_audio = load_silero_vad()

    # Step 2 — Get real VAD segments in original file sample space
    raw_vad_segments = segment_with_vad(
        audio_path, vad_model, get_speech_timestamps, read_audio
    )
    print(f"VAD found {len(raw_vad_segments)} raw segments")

    # Step 3 — Preprocess audio (resample → mono → trim silence → normalize)
    audio, sr = preprocess_audio(audio_path)
    print(f"Preprocessed audio length: {len(audio)/sr:.2f}s  ({len(audio)} samples)")

    # ── FIX: use real VAD positions; filter short segments (<0.3s) ──────────
    segments_with_indices = [
        (s, e) for s, e in raw_vad_segments if (e - s) >= sr * 0.3
    ]
    print(f"After short-segment filter: {len(segments_with_indices)} segments")

    # Safety: clamp segment boundaries to preprocessed audio length
    # (trim_silence may shorten the array slightly vs. the raw file)
    audio_len = len(audio)
    segments_with_indices = [
        (min(s, audio_len), min(e, audio_len))
        for s, e in segments_with_indices
    ]
    # ────────────────────────────────────────────────────────────────────────

    if not segments_with_indices:
        print("⚠  No valid segments found — skipping.")
        return [], sr

    # Step 4 — Merge + chunk
    chunked = merge_and_chunk_segments(segments_with_indices, max_duration=25.0, sr=sr)
    print(f"After merge+chunk:  {len(chunked)} chunks")

    # Step 5 — Add 2 s overlap between chunks
    chunked = add_overlap(chunked, sr=sr, overlap_sec=1.0)

    # Step 6 — Add boundary padding
    chunked = pad_chunks(chunked, audio_len=audio_len, sr=sr, pad_start=0.5, pad_end=0.4)
    print(f"Final chunk count:  {len(chunked)}")

    # Diagnostics — show last 3 chunks vs. audio tail
    print("\n--- Last 3 chunks ---")
    for i, (s, e) in enumerate(chunked[-3:]):
        idx = len(chunked) - 3 + i
        print(f"  chunk_{idx:03d}: [{s}:{e}]  {(e-s)/sr:.2f}s")
    print(f"  Audio tail sample: {audio_len}  ({audio_len/sr:.2f}s)")
    last_end = chunked[-1][1]
    tail_gap = audio_len - last_end
    print(f"  Uncovered tail: {tail_gap} samples  ({tail_gap/sr:.3f}s)")

    # Step 7 — Save chunks
    save_chunks(audio, chunked, sr=sr, output_dir=output_dir, file_id=file_id)

    return chunked, sr



In [ ]:
import os
import traceback

# =====================
# 6. ENTRY POINT
# =====================

if __name__ == "__main__":
    audio_files = sorted(os.listdir(TEST_AUDIO_DIR))
    print(f"🔍 Found {len(audio_files)} total files in {TEST_AUDIO_DIR}")

    for idx, fname in enumerate(audio_files):
        if not fname.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a")):
            print(f"⏭️ Skipping '{fname}' (does not match allowed extensions)")
            continue

        audio_path = os.path.join(TEST_AUDIO_DIR, fname)
        stem       = os.path.splitext(fname)[0]          # e.g. "test_001"
        file_id    = stem                                # used as chunk prefix

        print(f"⚙️ Processing: '{fname}' ...")
        
        try:
            chunks, sr = run_pipeline(
                audio_path=audio_path,
                file_id=file_id,
                output_dir=os.path.join(OUTPUT_CHUNK_DIR, stem)
            )
            print(f"✅ Successfully finished '{fname}'")
            
        except Exception as e:
            # If run_pipeline crashes, this catches it and prevents the loop from breaking
            print(f"❌ Error while processing '{fname}': {e}")
            traceback.print_exc() 

    print("\n🏁 All files processed.")

🔍 Found 24 total files in /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio
⚙️ Processing: 'test_001.wav' ...

 Processing: test_001.wav
Loading Silero VAD...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)
VAD found 422 raw segments
Preprocessed audio length: 3369.09s  (53905408 samples)
After short-segment filter: 422 segments
After merge+chunk:  362 chunks
Final chunk count:  362

--- Last 3 chunks ---
  chunk_359: [52007296:52063680]  3.52s
  chunk_360: [52068736:52137920]  4.32s
  chunk_361: [52132224:52200896]  4.29s
  Audio tail sample: 53905408  (3369.09s)
  Uncovered tail: 1704512 samples  (106.532s)
  Saved: test_001_chunk_000.wav  (3.61s,  samples [990208:1048000])
  Saved: test_001_chunk_001.wav  (3.68s,  samples [1054592:1113536])
  Saved: test_001_chunk_002.wav  (3.08s,  samples [1090944:114016

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)
VAD found 243 raw segments
Preprocessed audio length: 2427.49s  (38839808 samples)
After short-segment filter: 243 segments
After merge+chunk:  235 chunks
Final chunk count:  235

--- Last 3 chunks ---
  chunk_232: [37944704:38016960]  4.52s
  chunk_233: [38074240:38145472]  4.45s
  chunk_234: [38120320:38193600]  4.58s
  Audio tail sample: 38839808  (2427.49s)
  Uncovered tail: 646208 samples  (40.388s)
  Saved: test_002_chunk_000.wav  (24.28s,  samples [482304:870848])
  Saved: test_002_chunk_001.wav  (3.49s,  samples [843648:899520])
  Saved: test_002_chunk_002.wav  (10.66s,  samples [880000:1050560])
  Saved: test_002_chunk_003.wav  (19.72s,  samples [1024384:1339840])
  Saved: test_002_chunk_004.wav  (11.88s,  samples [1313152:1503168])
  Saved: test_002_chunk_005.wav  (26.90s,  samples [1474944:1905344])
  Saved: test_002_chunk_006.wav  (16.96s,  samples [1874944:2146240])
  Saved: test_002_chunk_007.wav  (16.55s,  samples [

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)
VAD found 627 raw segments
Preprocessed audio length: 3897.57s  (62361088 samples)
After short-segment filter: 627 segments
After merge+chunk:  482 chunks
Final chunk count:  482

--- Last 3 chunks ---
  chunk_479: [61577728:62008128]  26.90s
  chunk_480: [61977728:62138304]  10.04s
  chunk_481: [62112128:62246336]  8.39s
  Audio tail sample: 62361088  (3897.57s)
  Uncovered tail: 114752 samples  (7.172s)
  Saved: test_003_chunk_000.wav  (5.98s,  samples [228352:324032])
  Saved: test_003_chunk_001.wav  (5.92s,  samples [299904:394688])
  Saved: test_003_chunk_002.wav  (3.56s,  samples [373632:430528])
  Saved: test_003_chunk_003.wav  (13.41s,  samples [412544:627136])
  Saved: test_003_chunk_004.wav  (14.63s,  samples [701312:935360])
  Saved: test_003_chunk_005.wav  (14.95s,  samples [908160:1147328])
  Saved: test_003_chunk_006.wav  (22.21s,  samples [1124224:1479616])
  Saved: test_003_chunk_007.wav  (12.00s,  samples [1457536

...

VAD found 649 raw segments
Preprocessed audio length: 4009.31s  (64148992 samples)
After short-segment filter: 649 segments
After merge+chunk:  506 chunks
Final chunk count:  506

--- Last 3 chunks ---
  chunk_503: [63123328:63199168]  4.74s
  chunk_504: [63187840:63544256]  22.28s
  chunk_505: [63568768:63782848]  13.38s
  Audio tail sample: 64148992  (4009.31s)
  Uncovered tail: 366144 samples  (22.884s)
  Saved: test_027_chunk_000.wav  (12.76s,  samples [16896:221120])
  Saved: test_027_chunk_001.wav  (6.95s,  samples [197504:308672])
  Saved: test_027_chunk_002.wav  (5.76s,  samples [283520:375744])
  Saved: test_027_chunk_003.wav  (26.90s,  samples [349056:779456])
  Saved: test_027_chunk_004.wav  (3.55s,  samples [749056:805824])
  Saved: test_027_chunk_005.wav  (7.46s,  samples [790912:910272])
  Saved: test_027_chunk_006.wav  (10.31s,  samples [885632:1050560])
  Saved: test_027_chunk_007.wav  (4.68s,  samples [1027456:1102272])
  Saved: test_027_chunk_008.wav  (24.39s,  sample

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)
VAD found 245 raw segments
Preprocessed audio length: 2407.97s  (38527488 samples)
After short-segment filter: 245 segments
After merge+chunk:  201 chunks
Final chunk count:  201

--- Last 3 chunks ---
  chunk_198: [37416832:37510592]  5.86s
  chunk_199: [37517184:37571008]  3.36s
  chunk_200: [37634432:38045120]  25.67s
  Audio tail sample: 38527488  (2407.97s)
  Uncovered tail: 482368 samples  (30.148s)
  Saved: test_029_chunk_000.wav  (5.08s,  samples [508416:589760])
  Saved: test_029_chunk_001.wav  (26.90s,  samples [564096:994496])
  Saved: test_029_chunk_002.wav  (6.65s,  samples [964096:1070528])
  Saved: test_029_chunk_003.wav  (7.59s,  samples [1048960:1170368])
  Saved: test_029_chunk_004.wav  (25.19s,  samples [1145216:1548224])
  Saved: test_029_chunk_005.wav  (20.23s,  samples [1524608:1848256])
  Saved: test_029_chunk_006.wav  (13.22s,  samples [1822080:2033600])
  Saved: test_029_chunk_007.wav  (2.98s,  samples [26

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


✓ Silero VAD loaded (lightweight, 32ms frames at 16kHz)


In [1]:
# # =====================
# # 6. ENTRY POINT
# # =====================

# if __name__ == "__main__":
#     # Process all audio files in TRAIN_AUDIO_DIR
#     audio_files = sorted(os.listdir(TEST_AUDIO_DIR))

#     for idx, fname in enumerate(audio_files):
#         if not fname.lower().endswith((".wav", ".mp3", ".flac", ".ogg", ".m4a")):
#             continue

#         audio_path = os.path.join(TEST_AUDIO_DIR, fname)
#         stem       = os.path.splitext(fname)[0]          # e.g. "test_001"
#         file_id    = stem                                  # used as chunk prefix

#         chunks, sr = run_pipeline(
#             audio_path=audio_path,
#             file_id=file_id,
#             output_dir=os.path.join(OUTPUT_CHUNK_DIR, stem)
#         )

#     print("\n✅ All files processed.")

In [17]:
# !zip -r my_archive.zip /kaggle/working/train_001

In [ ]:
from rapidfuzz import fuzz
import re

def transcribe_chunks_to_list(audio, chunks, sr=16000):
    """Returns list of (chunk_index, text) instead of joined string."""
    results = []
    for i, (s, e) in enumerate(chunks):
        chunk = audio[s:e].astype(np.float32)
        inputs = processor(chunk, sampling_rate=sr, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        with torch.no_grad():
            predicted_ids = model.generate(input_features)
        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        results.append(text)
        print(f"Chunk {i:03d}: {text}")
    return results

In [ ]:
import os
import torch
import librosa
import pandas as pd
from rapidfuzz import fuzz
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# ── 1. Load model ──────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"

print("⏳ Loading model...")
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device)
model.eval()
print("✅ Model loaded.")

# ── 2. Define Stitching Function ───────────────────────────────
def stitch_chunks_fuzzy(chunk_texts, overlap_words=20, threshold=72):
    if not chunk_texts:
        return ""

    result_words = chunk_texts[0].split()

    for i in range(1, len(chunk_texts)):
        current_words = chunk_texts[i].split()
        best_score = 0
        best_cut_result  = 0
        
        for overlap_len in range(overlap_words, 2, -1):
            if overlap_len > len(result_words) or overlap_len > len(current_words):
                continue

            tail  = " ".join(result_words[-overlap_len:])
            head  = " ".join(current_words[:overlap_len])
            score = fuzz.ratio(tail, head)

            if score > best_score:
                best_score      = score
                best_cut_result = overlap_len

        if best_score >= threshold:
            result_words = result_words[:-best_cut_result] + current_words
        else:
            result_words = result_words + current_words

    return " ".join(result_words)

# ── 3. Iterate over all folders and transcribe ─────────────────
# Point this to the main folder containing your chunked subfolders
BASE_CHUNK_DIR = "/kaggle/working" 
final_results = []

# Get all subdirectories (e.g., test_001, test_002) in the base directory
folder_names = sorted([d for d in os.listdir(BASE_CHUNK_DIR) 
                       if os.path.isdir(os.path.join(BASE_CHUNK_DIR, d)) 
                       and d.startswith("test_")])

print(f"🔍 Found {len(folder_names)} folders to process.")

for folder_name in folder_names:
    chunk_dir = os.path.join(BASE_CHUNK_DIR, folder_name)
    chunk_files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".wav")])
    
    if not chunk_files:
        continue
        
    print(f"⚙️ Transcribing {len(chunk_files)} chunks for {folder_name}...")
    
    chunk_texts = []
    for fname in chunk_files:
        path = os.path.join(chunk_dir, fname)
        audio, sr = librosa.load(path, sr=16000)

        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(input_features)

        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        chunk_texts.append(text)

    # Stitch the transcribed chunks together
    full_text = stitch_chunks_fuzzy(chunk_texts, overlap_words=20, threshold=75)
    
    # Save to our results list
    final_results.append({
        "filename": folder_name, # e.g., "test_001"
        "transcript": full_text
    })
    print(f"✅ Finished {folder_name}")

# ── 4. Save to CSV ─────────────────────────────────────────────
df = pd.DataFrame(final_results)
csv_output_path = "submission.csv"
df.to_csv(csv_output_path, index=False)

print(f"\n🎉 All done! Saved {len(df)} transcripts to {csv_output_path}")
print(df.head())

In [41]:
# import os
# import torch
# import librosa
# import numpy as np
# from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# # ── Load model ──────────────────────────────────────────────
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_id = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"

# processor = AutoProcessor.from_pretrained(model_id)
# model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id).to(device)
# model.eval()

# # ── Transcribe from saved .wav files ────────────────────────
# chunk_dir = "/kaggle/working/test_001"
# chunk_files = sorted([f for f in os.listdir(chunk_dir) if f.endswith(".wav")])

# results = []
# for fname in chunk_files:
#     path = os.path.join(chunk_dir, fname)
#     audio, sr = librosa.load(path, sr=16000)

#     inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
#     input_features = inputs.input_features.to(device)

#     with torch.no_grad():
#         predicted_ids = model.generate(input_features)

#     text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
#     print(f"{fname}: {text}")
#     results.append({"file": fname, "text": text})


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

train_001_chunk_000.wav: গল্পতরু চ্যানেলে আপনাদের সাথে আছি আমি রাজ পড়ছিলাম হুমায়ূন আহমেদের উপন্যাস মধ্যাহ্ন পড়ছি এর চতুর্থ পর্ব
train_001_chunk_001.wav: রঙিলা নটিবাড়ি সোহাবগঞ্জ বাজারে শেষ মাথায় মাছের আড়ত পার হয়েও আট দশ মিনিট হাঁটতে হয়
train_001_chunk_002.wav: রাস্তার দুপাশে আপনাতে গজিয়ে ওঠা বেশ কিছু শিমুল গাছ যে কেউ দেখে ভাববে কোনো এক বৃক্ষপ্রেমী চিন্তা ভাবনা করে শিমুলের শাড়ি লাগিয়েছেন
train_001_chunk_003.wav: চৈত্র মাসে শিমুলের টকটকে লাল ফুল ফোটে দেখতে ভালো লাগে মনে হয় চৈত্রের তীব্র উত্তাপে গাছের মাথায় আগুন লেগে গেছে
train_001_chunk_004.wav: মূল বাড়ি কাঠের উপরের টিন মূল বাড়ি ঘিরে এক রুমের বেশ কিছু ছোট ছোট ঘর কাঠের মূল বাড়িটা দর্শনীয় উচ্চতায় প্রায় দোতলা বাড়ি সমান দরজা এবং পাল্লায় ফুল লতাপাতা আঁকা টিনের চৌচালাতেও নকশা কাটা লক্ষ্ণয়সের বাইজি আঙ্গুরি অনেক টাকা পয়সা খরচ করে মূল বাড়ি তৈরি করে এই বাড়িতে তারা
train_001_chunk_005.wav:  এই বাড়িতে তার একটা কন্যা সন্তান হয় তার নাম বেদানা তিন বছর বয়সে বেদানা পানিতে ডুবে মারা যায় বেদানার মৃত্যুর পর আঙ্গুরের আর কোন খোঁজ পাওয়া যায়নি কেউ বলে আঙ্গুরেও ম

NameError: name 'chunk_texts' is not defined

In [44]:
# chunk_texts = []
# for fname in chunk_files:
#     path  = os.path.join(chunk_dir, fname)
#     audio, sr = librosa.load(path, sr=16000)
#     inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
#     input_features = inputs.input_features.to(device)
#     with torch.no_grad():
#         predicted_ids = model.generate(input_features)
#     text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
#     print(f"{fname}: {text}")
#     chunk_texts.append(text)

train_001_chunk_000.wav: গল্পতরু চ্যানেলে আপনাদের সাথে আছি আমি রাজ পড়ছিলাম হুমায়ূন আহমেদের উপন্যাস মধ্যাহ্ন পড়ছি এর চতুর্থ পর্ব
train_001_chunk_001.wav: রঙিলা নটিবাড়ি সোহাবগঞ্জ বাজারে শেষ মাথায় মাছের আড়ত পার হয়েও আট দশ মিনিট হাঁটতে হয়
train_001_chunk_002.wav: রাস্তার দুপাশে আপনাতে গজিয়ে ওঠা বেশ কিছু শিমুল গাছ যে কেউ দেখে ভাববে কোনো এক বৃক্ষপ্রেমী চিন্তা ভাবনা করে শিমুলের শাড়ি লাগিয়েছেন
train_001_chunk_003.wav: চৈত্র মাসে শিমুলের টকটকে লাল ফুল ফোটে দেখতে ভালো লাগে মনে হয় চৈত্রের তীব্র উত্তাপে গাছের মাথায় আগুন লেগে গেছে
train_001_chunk_004.wav: মূল বাড়ি কাঠের উপরের টিন মূল বাড়ি ঘিরে এক রুমের বেশ কিছু ছোট ছোট ঘর কাঠের মূল বাড়িটা দর্শনীয় উচ্চতায় প্রায় দোতলা বাড়ি সমান দরজা এবং পাল্লায় ফুল লতাপাতা আঁকা টিনের চৌচালাতেও নকশা কাটা লক্ষ্ণয়সের বাইজি আঙ্গুরি অনেক টাকা পয়সা খরচ করে মূল বাড়ি তৈরি করে এই বাড়িতে তারা
train_001_chunk_005.wav:  এই বাড়িতে তার একটা কন্যা সন্তান হয় তার নাম বেদানা তিন বছর বয়সে বেদানা পানিতে ডুবে মারা যায় বেদানার মৃত্যুর পর আঙ্গুরের আর কোন খোঁজ পাওয়া যায়নি কেউ বলে আঙ্গুরেও ম

In [68]:
# from rapidfuzz import fuzz
# import re

# def stitch_chunks_fuzzy(chunk_texts, overlap_words=20, threshold=72):
#     if not chunk_texts:
#         return ""

#     result_words = chunk_texts[0].split()

#     for i in range(1, len(chunk_texts)):
#         current_words = chunk_texts[i].split()

#         best_score = 0
#         best_cut_result  = 0  # how many words to drop from end of result
#         best_cut_current = 0  # how many words to drop from start of current

#         for overlap_len in range(overlap_words, 2, -1):
#             if overlap_len > len(result_words) or overlap_len > len(current_words):
#                 continue

#             tail  = " ".join(result_words[-overlap_len:])
#             head  = " ".join(current_words[:overlap_len])
#             score = fuzz.ratio(tail, head)

#             if score > best_score:
#                 best_score       = score
#                 best_cut_result  = overlap_len   # drop from chunk N tail
#                 best_cut_current = overlap_len   # drop from chunk N+1 head

#         if best_score >= threshold:
#             # ── KEY CHANGE ──────────────────────────────────────────
#             # Drop overlapping tail from chunk N (less reliable region)
#             # Keep full start of chunk N+1 (more reliable region)
#             result_words = result_words[:-best_cut_result] + current_words
#             # ────────────────────────────────────────────────────────
#         else:
#             result_words = result_words + current_words

#     return " ".join(result_words)

In [69]:
# # ── Full transcript ──────────────────────────────────────────
# full_text = stitch_chunks_fuzzy(chunk_texts, overlap_words=20, threshold=75)
# print("\n── Full Transcript ──")
# print(full_text)


── Full Transcript ──
গল্পতরু চ্যানেলে আপনাদের সাথে আছি আমি রাজ পড়ছিলাম হুমায়ূন আহমেদের উপন্যাস মধ্যাহ্ন পড়ছি এর চতুর্থ পর্ব রঙিলা নটিবাড়ি সোহাবগঞ্জ বাজারে শেষ মাথায় মাছের আড়ত পার হয়েও আট দশ মিনিট হাঁটতে হয় রাস্তার দুপাশে আপনাতে গজিয়ে ওঠা বেশ কিছু শিমুল গাছ যে কেউ দেখে ভাববে কোনো এক বৃক্ষপ্রেমী চিন্তা ভাবনা করে শিমুলের শাড়ি লাগিয়েছেন চৈত্র মাসে শিমুলের টকটকে লাল ফুল ফোটে দেখতে ভালো লাগে মনে হয় চৈত্রের তীব্র উত্তাপে গাছের মাথায় আগুন লেগে গেছে মূল বাড়ি কাঠের উপরের টিন মূল বাড়ি ঘিরে এক রুমের বেশ কিছু ছোট ছোট ঘর কাঠের মূল বাড়িটা দর্শনীয় উচ্চতায় প্রায় দোতলা বাড়ি সমান দরজা এবং পাল্লায় ফুল লতাপাতা আঁকা টিনের চৌচালাতেও নকশা কাটা লক্ষ্ণয়সের বাইজি আঙ্গুরি অনেক টাকা পয়সা খরচ করে মূল বাড়ি তৈরি করে এই বাড়িতে তার একটা কন্যা সন্তান হয় তার নাম বেদানা তিন বছর বয়সে বেদানা পানিতে ডুবে মারা যায় বেদানার মৃত্যুর পর আঙ্গুরের আর কোন খোঁজ পাওয়া যায়নি কেউ বলে আঙ্গুরেও মের মত পানিতে ডুবে গেছে আবার কারো কারো মতে আঙ্গুরের দেশান্তরী হয়েছে দেশান্তরী শব্দটা এলাকার মানুষের অতি প্রিয় স্ত্রীর সঙ্গে ঝগড়া হলেই স্বামীরা ব